In [0]:
empdata = [[101,'Rahul','pune'],[102,'Akash','mumbai']]

In [0]:
df = spark.createDataFrame(empdata,['id','name','city'])

In [0]:
df.show()

+---+-----+------+
| id| name|  city|
+---+-----+------+
|101|Rahul|  pune|
|102|Akash|mumbai|
+---+-----+------+



In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("employee")

In [0]:
spark.sql("describe formatted employee").show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|                  id|              bigint|   NULL|
|                name|              string|   NULL|
|                city|              string|   NULL|
|                    |                    |       |
|# Delta Statistic...|                    |       |
|        Column Names|      id, name, city|       |
|Column Selection ...|            first-32|       |
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|online1012ws_7405...|       |
|            Database|             default|       |
|               Table|            employee|       |
|        Created Time|Tue Jun 02 15:30:...|       |
|         Last Access|             UNKNOWN|       |
|          Created By|              Spark |       |
|                Type|             MANAGED|       |
|           

In [0]:
%sql

insert into employee values(500,'test','sample')

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql 

select * from employee

id,name,city
500,test,sample
101,Rahul,pune
102,Akash,mumbai


In [0]:
df.write.format("delta").mode("overwrite").save("abfss://projectdata@online1012stgaccount.dfs.core.windows.net/prjdata")

In [0]:
spark.sql("create table empext using delta location 'abfss://projectdata@online1012stgaccount.dfs.core.windows.net/prjdata' ")

DataFrame[]

In [0]:
spark.sql("describe formatted empext").show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|                  id|              bigint|   NULL|
|                name|              string|   NULL|
|                city|              string|   NULL|
|                    |                    |       |
|# Delta Statistic...|                    |       |
|        Column Names|      id, name, city|       |
|Column Selection ...|            first-32|       |
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|online1012ws_7405...|       |
|            Database|             default|       |
|               Table|              empext|       |
|        Created Time|Tue Jun 02 15:31:...|       |
|         Last Access|             UNKNOWN|       |
|          Created By|              Spark |       |
|                Type|            EXTERNAL|       |
|           

In [0]:
%sql

insert into empext values(501,'test','sample')

num_affected_rows,num_inserted_rows
1,1


In [0]:
spark.conf.set("spark.databricks.delta.deletedFileRetentionDuration.enabled","False")

In [0]:
%sql

optimize empext
zorder by id

path,metrics
abfss://projectdata@online1012stgaccount.dfs.core.windows.net/prjdata,"List(1, 4, List(1338, 1338, 1338.0, 1, 1338), List(1196, 1198, 1197.5, 4, 4790), 0, List(minCubeSize(107374182400), List(0, 0), List(4, 4790), 0, List(4, 4790), 1, null), null, 0, 1, 4, 0, false, 0, 0, 1780414558920, 1780414566692, 4, 1, null, List(0, 0), null, 3, 3, 1139, 0, null, null)"


In [0]:
spark.sql("SET delta.retentionDurationCheck.enabled = false")



DataFrame[key: string, value: string]

In [0]:
spark.sql("VACUUM empext RETAIN 1 HOURS")

DataFrame[path: string]